# 06 · 出力整合（output matching）による切片推定

実験 A で、LOSO-B の切片ずれは「症例構成による押し上げ」と「撮像由来の押し下げ」が綱引きした合成であり、
field では有病率がほぼ不変のまま撮像だけで融合ロジットが −0.33 動くことが分かった。
比方式（間接標準化：同じ π モデルで target と source の両方を引く）が拾えるのは症例構成由来の分だけ。

出力整合は、ずれの原因を問わず総量を一括で推定する：

$$\text{find } \hat b \text{ such that } \frac1N\sum_i \sigma(z_i + \hat b) = \hat\pi_{\text{cov}}$$

- $z_i = \log(\alpha_i/\beta_i)$：補正前の融合ロジット
- $\hat\pi_{\text{cov}}$：train 側でフィットした π(年齢, 疑い) を target の共変量に当てた有病率。疑いが不明な症例は年齢のみの π
- 左辺は b について単調増加なので、brentq で一意に解ける（bracket [−10, 10]。到達できない場合は端に clip して警告）

EM との違い：有病率はモデルを通らない情報（共変量）だけで決め、モデル出力は「ずれの測定」にだけ使う。循環が無い。

**実装の検証**：係留点を真の有病率にした出力整合は、全例ラベルでの切片 MLE と同じスコア方程式
$\sum_i\sigma(z_i+b)=\sum_i y_i$ を満たすので、数値的に一致しなければならない（手法 e と f）。

ロジックは `openidh_model/eval/output_matching.py` にある。この notebook は呼んで記録するだけ。
入力は `runs/*/predictions.csv` と `data/metadata` のみ（推論なし、torch なし）。

In [1]:
reps       = 50                 # evaluations 2 and 3: draws per run
seed       = 0
ns         = [5, 10, 20, 30, 50, 100, 200]
flip_rate  = 0.15
missing    = [0.0, 0.25, 0.5, 1.0]
harm_threshold = 0.10
out_json   = "results/output_matching.json"
out_md     = "results/output_matching.md"
out_fig    = "results/figures/output_matching_n"

In [2]:
import os, sys, json, warnings
from pathlib import Path
_root = Path.cwd()
while not (_root / "data" / "metadata").exists():
    _root = _root.parent
os.chdir(_root); sys.path.insert(0, str(_root))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
from openidh_model.utils.paths import load_paths
from openidh_model.eval import output_matching as om

for f in (out_json, out_md, out_fig + ".png", out_fig + ".svg"):
    assert not Path(f).exists() or Path(f).stat().st_size == 0 or os.environ.get("OM_OVERWRITE") == "1", \
        f"{f} exists — refusing to overwrite an existing result (set OM_OVERWRITE=1 to rerun)"

UNITS = ["LOSO-A", "LOSO-B", "field 3T", "vendor Philips", "in-dist"]
paths = load_paths(os.environ.get("OPENIDH_PATHS", "configs/paths.local.yaml"))
runs = om.load_runs(paths, "runs")
print(f"{len(runs)} runs:", pd.Series([r.unit for r in runs]).value_counts().reindex(UNITS).to_dict())

27 runs: {'LOSO-A': 3, 'LOSO-B': 3, 'field 3T': 3, 'vendor Philips': 3, 'in-dist': 15}


## 0. 検算

- e（oracle 係留の出力整合）と f（全例ラベルの切片 MLE、別実装で NLL を直接最小化）が一致すること
- a〜f のすべてで AUC の変化が 0 であること（施設一律の定数切片なので）
- bracket 端への張り付き（clip）が無いこと。あれば除外せず、ここに列挙する
- LOSO-B の $\hat\pi_{\text{cov}}$ が既報の 27.8%（grade_prior_eval 水準 B）と一致すること

In [3]:
with warnings.catch_warnings(record=True) as w1:
    warnings.simplefilter("always")
    full = om.evaluate_full(runs)
b = full.pivot_table(index="run", columns="method", values="b")
auc = full.pivot_table(index="run", columns="method", values="auc")
checks = {
    "max_abs_e_minus_f": float((b["e_om_oracle"] - b["f_label_mle"]).abs().max()),
    "max_abs_dAUC": float(auc.sub(auc["a_raw"], axis=0).abs().max().max()),
    "clipped_eval1": full.loc[full.status != "ok", ["run", "method", "status"]].to_dict("records"),
    "warnings_eval1": [str(x.message) for x in w1],
    "pi_cov_test_lossb": float(full[full.unit == "LOSO-B"]["pi_cov_test"].mean()),
}
print(f"max |b_e - b_f| = {checks['max_abs_e_minus_f']:.2e}   (must be ~0)")
print(f"max |ΔAUC|      = {checks['max_abs_dAUC']:.2e}   (must be 0)")
print(f"clipped runs    = {checks['clipped_eval1'] or 'none'}")
print(f"LOSO-B π̂_cov    = {checks['pi_cov_test_lossb']*100:.2f}%  (既報 27.8%)")
assert checks["max_abs_e_minus_f"] < 1e-5 and checks["max_abs_dAUC"] < 1e-12

max |b_e - b_f| = 3.07e-08   (must be ~0)
max |ΔAUC|      = 0.00e+00   (must be 0)
clipped runs    = none
LOSO-B π̂_cov    = 27.87%  (既報 27.8%)


## 1. 評価 1：全例での性能比較（27 ラン）

同じ test 集合でのペア比較。値は各ユニットのラン平均（OOD は 3 seed、分布内は 5 fold × 3 seed = 15 ラン）。
b0（比方式・年齢のみ）は処方箋の階段を作るために加えた参考行。

In [4]:
MORDER = list(om.METHODS_FULL)
full["unit"] = pd.Categorical(full["unit"], UNITS, ordered=True)
full["method"] = pd.Categorical(full["method"], MORDER, ordered=True)
g = full.groupby(["unit", "method"], observed=True)
t1 = g[["ece", "brier", "nll", "sens", "spec", "auc"]].mean()
t1["b_mean"] = g["b"].mean(); t1["b_sd"] = g["b"].std(ddof=1)
t1 = t1.reset_index()
t1["手法"] = t1["method"].map(om.METHODS_FULL)
display(t1.pivot_table(index="手法", columns="unit", values="ece", observed=True, sort=False)
          .reindex([om.METHODS_FULL[m] for m in MORDER]).round(3))
display(t1.round(3))

unit,LOSO-A,LOSO-B,field 3T,vendor Philips,in-dist
手法,,,,,
補正なし,0.082,0.162,0.088,0.085,0.044
比方式（年齢のみ）,0.074,0.105,0.091,0.096,0.045
比方式（年齢+疑い）,0.070,0.064,0.090,0.092,0.044
出力整合（年齢+疑い）,0.038,0.042,0.070,0.092,0.044
出力整合（年齢のみ）,0.037,0.090,0.074,0.100,0.045
出力整合（oracle 係留）,0.042,0.045,0.078,0.086,0.043
全例ラベル切片,0.042,0.045,0.078,0.086,0.043


,unit,method,ece,brier,nll,sens,spec,auc,b_mean,b_sd,手法
0,LOSO-A,a_raw,0.082,0.122,0.400,0.262,0.982,0.856,0.000,0.000,補正なし
1,LOSO-A,b0_ratio_age,0.074,0.120,0.393,0.282,0.982,0.856,0.088,0.000,比方式（年齢のみ）
2,LOSO-A,b_ratio,0.070,0.120,0.391,0.294,0.982,0.856,0.125,0.000,比方式（年齢+疑い）
3,LOSO-A,c_om_cov,0.038,0.111,0.361,0.450,0.962,0.856,0.775,0.347,出力整合（年齢+疑い）
4,LOSO-A,d_om_age,0.037,0.111,0.361,0.450,0.959,0.856,0.783,0.347,出力整合（年齢のみ）
5,LOSO-A,e_om_oracle,0.042,0.110,0.360,0.460,0.958,0.856,0.844,0.345,出力整合（oracle 係留）
6,LOSO-A,f_label_mle,0.042,0.110,0.360,0.460,0.958,0.856,0.844,0.345,全例ラベル切片
7,LOSO-B,a_raw,0.162,0.177,0.559,0.102,0.992,0.854,0.000,0.000,補正なし
8,LOSO-B,b0_ratio_age,0.105,0.153,0.478,0.316,0.966,0.854,0.601,0.000,比方式（年齢のみ）
9,LOSO-B,b_ratio,0.064,0.141,0.442,0.492,0.928,0.854,1.065,0.000,比方式（年齢+疑い）


### 事前予測との照合

仕様書に書いた事前予測（出力整合 c の ECE）と、実測の差を並べる。外れた場合は外れ方を記す。
仕様書に「比方式 ECE」として挙げた LOSO-B 0.062 / LOSO-A 0.074 は CBTRUS 外部表による比方式の値。
ここでの b は train 内部フィットの π を使っているので、実測の b 列と一致するとは限らない。

In [5]:
E = t1.set_index(["unit", "method"])
def ece(u, m): return float(E.loc[(u, m), "ece"])
def bb(u, m):  return float(E.loc[(u, m), "b_mean"])
pred = [
    ("LOSO-B", "0.046–0.052", 0.046, 0.052),
    ("LOSO-A", "0.045–0.058", 0.045, 0.058),
    ("field 3T", "0.078 前後（±0.005）、b̂ が正で +0.3 前後", 0.073, 0.083),
    ("vendor Philips", "比方式と同等か微改善", None, None),
    ("in-dist", "不変（b̂ ≈ 0）", None, None),
]
rows = []
for u, txt, lo, hi in pred:
    c = ece(u, "c_om_cov"); r = ece(u, "b_ratio"); a = ece(u, "a_raw"); bc = bb(u, "c_om_cov")
    if lo is not None:
        verdict = "的中" if lo <= c <= hi else ("予測より良い" if c < lo else "予測より悪い")
    elif u == "vendor Philips":
        verdict = "的中" if c <= r + 0.002 else "外れ（比方式より悪い）"
        verdict += "／補正なしより悪化" if c > a else ""
    else:
        verdict = "的中" if abs(c - a) < 0.005 and abs(bc) < 0.1 else "外れ"
    rows.append({"unit": u, "予測": txt, "補正なし": a, "比方式(内部)": r, "出力整合 c": c,
                 "b̂_c": bc, "oracle e": ece(u, "e_om_oracle"), "判定": verdict})
prereg = pd.DataFrame(rows)
display(prereg.round(3))

,unit,予測,補正なし,比方式(内部),出力整合 c,b̂_c,oracle e,判定
0,LOSO-B,0.046–0.052,0.162,0.064,0.042,1.415,0.045,予測より良い
1,LOSO-A,0.045–0.058,0.082,0.070,0.038,0.775,0.042,予測より良い
2,field 3T,0.078 前後（±0.005）、b̂ が正で +0.3 前後,0.088,0.090,0.070,0.198,0.078,予測より良い
3,vendor Philips,比方式と同等か微改善,0.085,0.092,0.092,0.049,0.086,的中／補正なしより悪化
4,in-dist,不変（b̂ ≈ 0）,0.044,0.044,0.044,0.034,0.043,的中


## 2. 評価 2：N 依存性

新しい施設に最初に来る N 例を模擬する。test から症例単位で N 例を抽出し、**同じ N 例**から

- 出力整合：N 例の共変量で $\hat\pi_{\text{cov}}$、N 例の予測値で b̂ を解く（ラベル不要）
- 比方式：N 例の共変量で $\hat\pi_{\text{cov}}$ を出し、source 側（val）の π̂ と比をとる（ラベル不要）
- ラベルあり：N 例のラベルで切片 MLE を出す（既存の「ラベル N 例」曲線。単一クラスの抽出は採点しない）

を推定し、**残りの症例**で ECE を測る。ラン当たり 50 回。OOD は 3 seed で 150 回、分布内は 15 ランで 750 回。
N 例を抜くと評価側が 20 例を切る組み合わせ（field の N=200 など）は計算しない。

**害の確率** = 補正なし（同じ残り症例での ECE）より悪化した反復の割合（仕様どおり）。
補足として **P(害 >0.01)** = ECE が 0.01 を超えて悪化した割合も出す。補正すべきずれが無いユニット（分布内）では、
b̂ が 0 のまわりの雑音になるので、どの手法でも仕様どおりの P(害) は 50% 前後に張り付く。これは害の大きさを区別しない。

**注意**：N を大きくすると残り症例が減り、ECE 自体が上に偏る（10 ビン ECE の小標本バイアス）。field（test 195 例）と vendor（140 例）の
N=100 は残りが 95 / 40 例しかなく、補正なしも含めてすべての手法の ECE が跳ね上がる。この行は手法比較には使えない。

In [6]:
with warnings.catch_warnings(record=True) as w2:
    warnings.simplefilter("always")
    ncurve = om.n_curve(runs, ns=ns, reps=reps, seed=seed)
summ = om.summarise_n(ncurve)
summ["unit"] = pd.Categorical(summ["unit"], UNITS, ordered=True)
summ = summ.sort_values(["unit", "n", "method"]).reset_index(drop=True)
clip2 = ncurve[ncurve.status.str.startswith("clip")].groupby(["unit", "n", "method"], observed=True).size()
print("bracket clips in evaluation 2:", "none" if clip2.empty else "")
if not clip2.empty: display(clip2.rename("clipped draws").reset_index())

bracket clips in evaluation 2: none


In [7]:
COL = {"om": "#2a78d6", "label": "#eb6834", "ratio": "#1baf7a"}
# figure text is English: the analysis boxes (Docker, HPC nodes) carry no CJK font
NAME = {"om": "output matching (N unlabelled)", "label": "labelled N (intercept MLE)", "ratio": "ratio (covariates of N)"}
REF = {"raw": ("uncorrected", (0, (1, 2))), "prior_oracle": ("prior oracle", (0, (6, 3))),
       "full_label": ("all labels (= oracle anchor)", "-")}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})
fig, axs = plt.subplots(2, 3, figsize=(12, 7.2), sharey=True)
axes = axs.ravel()
for ax, u in zip(axes, UNITS):
    s = summ[summ.unit == u]
    # references also vary with N: they are scored on the same shrinking remainder
    for m, (lab, ls) in REF.items():
        r = s[s.method == m]
        ax.plot(r["n"], r["ece_mean"], color="#8a8a86", lw=1.2, ls=ls, label=lab, zorder=1)
    for m in ("om", "label", "ratio"):
        r = s[s.method == m]
        ax.plot(r["n"], r["ece_mean"], color=COL[m], lw=2, marker="o", ms=4, label=NAME[m], zorder=3)
        if m == "om":
            ax.fill_between(r["n"], r["ece_lo"], r["ece_hi"], color=COL[m], alpha=.12, lw=0, zorder=2,
                            label="95% interval (output matching)")
    ax.set_xscale("log"); ax.set_xticks(ns); ax.set_xticklabels(ns, fontsize=8); ax.minorticks_off()
    ax.set_title(u, fontsize=10, loc="left"); ax.set_xlabel("N (first cases at the site)", fontsize=8)
    ax.grid(axis="y", color="#e6e5e0", lw=.6)
    ax.tick_params(labelleft=True)
for ax in axs[:, 0]:
    ax.set_ylabel("ECE on remaining cases (mean)")
h, l = axes[0].get_legend_handles_labels()
axes[-1].axis("off")
axes[-1].legend(h, l, fontsize=9, frameon=False, loc="center left")
fig.suptitle("ECE vs N: first N cases at a new site, scored on the rest (50 draws x run)",
             fontsize=11, x=0.01, ha="left")
fig.tight_layout()
Path(out_fig).parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_fig + ".png", dpi=160); fig.savefig(out_fig + ".svg")
plt.show()

### N 表（ユニット別）

ece は残り症例での平均 [2.5–97.5 百分位]、b_sd は推定切片の反復間 sd、P(害) は補正なしより悪化した割合。

In [8]:
def ntable(u):
    s = summ[(summ.unit == u) & summ.method.isin(["om", "ratio", "label"])].copy()
    s["ece"] = s.apply(lambda r: f"{r.ece_mean:.3f} [{r.ece_lo:.3f}–{r.ece_hi:.3f}]", axis=1)
    s["P(害)"] = (s["p_harm"] * 100).round(0).astype("Int64").astype(str) + "%"
    s["P(害>0.01)"] = (s["p_harm_tol"] * 100).round(0).astype("Int64").astype(str) + "%"
    w = s.pivot_table(index="n", columns="method", values=["ece", "b_sd", "P(害)", "P(害>0.01)"], aggfunc="first", observed=True)
    return w.reindex(columns=["om", "ratio", "label"], level=1)
for u in UNITS:
    display(Markdown(f"**{u}**")); display(ntable(u).round(3))

**LOSO-A**

P(害)             P(害>0.01)               b_sd                                ece                                          
method   om ratio label        om ratio label     om  ratio  label                   om                ratio                label
n                                                                                                                                
5       59%   55%   49%       51%   47%   44%  1.204  1.274  0.946  0.102 [0.026–0.223]  0.104 [0.025–0.196]  0.121 [0.025–0.350]
10      37%   51%   44%       28%   43%   33%  0.869  0.775  0.816  0.075 [0.025–0.186]  0.088 [0.025–0.176]  0.087 [0.026–0.212]
20      23%   39%   41%       17%   29%   33%  0.608  0.500  0.771  0.061 [0.025–0.131]  0.077 [0.025–0.154]  0.075 [0.026–0.170]
30      19%   41%   21%        9%   23%   15%  0.467  0.355  0.553  0.056 [0.025–0.100]  0.074 [0.026–0.132]  0.061 [0.027–0.147]
50      12%   35%   21%        5%   14%   14%  0.440  0.288  0.511  0.052 [0.026–0.091]  0.073 [0.023–0.124]  0.059 [0.026–0.121]
100      6%   33%   12%        2%   15%    8%  0.348  0.208  0.402  0.049 [0.025–0.078]  0.076 [0.029–0.138]  0.052 [0.025–0.098]
200      3%   12%   10%        2%    0%    5%  0.311  0.108  0.323  0.048 [0.027–0.079]  0.076 [0.035–0.124]  0.052 [0.027–0.086]

**LOSO-B**

P(害)             P(害>0.01)               b_sd                                ece                                          
method   om ratio label        om ratio label     om  ratio  label                   om                ratio                label
n                                                                                                                                
5       24%   21%   30%       21%   21%   27%  1.155  1.205  0.925  0.115 [0.039–0.262]  0.114 [0.038–0.272]  0.130 [0.041–0.339]
10      11%    8%   19%       10%    5%   17%  0.756  0.715  0.871  0.086 [0.037–0.209]  0.087 [0.038–0.200]  0.106 [0.041–0.250]
20       1%    2%    8%        1%    1%    6%  0.486  0.418  0.666  0.065 [0.038–0.131]  0.076 [0.039–0.149]  0.078 [0.038–0.192]
30       0%    0%    1%        0%    0%    1%  0.421  0.332  0.506  0.062 [0.037–0.120]  0.072 [0.038–0.134]  0.069 [0.041–0.129]
50       0%    0%    1%        0%    0%    1%  0.376  0.266  0.427  0.059 [0.036–0.106]  0.070 [0.038–0.127]  0.063 [0.039–0.133]
100      0%    0%    0%        0%    0%    0%  0.302  0.191  0.321  0.055 [0.036–0.082]  0.069 [0.040–0.124]  0.058 [0.037–0.093]
200      0%    0%    0%        0%    0%    0%  0.231  0.107  0.253  0.056 [0.038–0.077]  0.066 [0.037–0.107]  0.057 [0.038–0.075]

**field 3T**

P(害)             P(害>0.01)               b_sd                                ece                                          
method   om ratio label        om ratio label     om  ratio  label                   om                ratio                label
n                                                                                                                                
5       69%   73%   72%       53%   57%   53%  1.138  0.924  1.032  0.118 [0.060–0.229]  0.118 [0.064–0.239]  0.132 [0.070–0.337]
10      59%   74%   64%       43%   54%   52%  0.839  0.606  0.935  0.103 [0.060–0.172]  0.110 [0.067–0.179]  0.112 [0.060–0.231]
20      52%   77%   61%       27%   52%   47%  0.694  0.443  0.871  0.094 [0.063–0.147]  0.108 [0.060–0.185]  0.108 [0.064–0.202]
30      45%   76%   55%       24%   58%   36%  0.582  0.296  0.639  0.091 [0.064–0.130]  0.107 [0.067–0.159]  0.096 [0.063–0.145]
50      40%   71%   51%       21%   41%   33%  0.508  0.197  0.593  0.088 [0.059–0.126]  0.099 [0.060–0.142]  0.095 [0.063–0.140]
100     45%   62%   50%       29%   49%   28%  0.452  0.144  0.479  0.102 [0.058–0.148]  0.111 [0.063–0.169]  0.103 [0.058–0.145]

**vendor Philips**

P(害)             P(害>0.01)               b_sd                                ece                                          
method   om ratio label        om ratio label     om  ratio  label                   om                ratio                label
n                                                                                                                                
5       86%   82%   93%       67%   65%   75%  0.930  0.895  1.113  0.130 [0.061–0.247]  0.125 [0.065–0.255]  0.147 [0.065–0.356]
10      84%   78%   84%       63%   52%   67%  0.686  0.533  0.889  0.114 [0.061–0.232]  0.105 [0.059–0.175]  0.129 [0.061–0.286]
20      81%   67%   83%       55%   39%   60%  0.491  0.384  0.620  0.108 [0.064–0.190]  0.100 [0.057–0.155]  0.115 [0.062–0.193]
30      78%   67%   71%       49%   39%   45%  0.401  0.280  0.477  0.105 [0.068–0.158]  0.099 [0.060–0.140]  0.107 [0.058–0.169]
50      66%   71%   70%       44%   39%   44%  0.282  0.183  0.347  0.106 [0.061–0.165]  0.105 [0.061–0.162]  0.108 [0.064–0.167]
100     65%   61%   56%       47%   47%   40%  0.186  0.079  0.223  0.145 [0.086–0.219]  0.145 [0.085–0.231]  0.143 [0.089–0.219]

**in-dist**

P(害)             P(害>0.01)               b_sd                                ece                                          
method   om ratio label        om ratio label     om  ratio  label                   om                ratio                label
n                                                                                                                                
5       85%   84%   85%       72%   65%   72%  1.323  1.268  1.145  0.090 [0.032–0.230]  0.085 [0.033–0.173]  0.113 [0.033–0.320]
10      79%   77%   84%       58%   55%   64%  0.928  0.763  0.951  0.072 [0.030–0.159]  0.065 [0.030–0.136]  0.082 [0.030–0.202]
20      70%   69%   75%       47%   41%   53%  0.707  0.508  0.822  0.060 [0.029–0.117]  0.056 [0.028–0.106]  0.067 [0.030–0.142]
30      69%   66%   74%       42%   37%   50%  0.603  0.401  0.706  0.057 [0.030–0.102]  0.055 [0.028–0.098]  0.063 [0.029–0.123]
50      63%   59%   68%       33%   25%   45%  0.516  0.295  0.576  0.054 [0.028–0.089]  0.051 [0.027–0.083]  0.058 [0.029–0.109]
100     53%   58%   60%       23%   17%   30%  0.427  0.193  0.418  0.054 [0.030–0.084]  0.053 [0.028–0.083]  0.056 [0.029–0.090]
200     49%   54%   50%       23%   15%   27%  0.374  0.133  0.328  0.066 [0.036–0.102]  0.066 [0.037–0.102]  0.067 [0.037–0.105]

### 運用ルールの導出

**害の確率が 10% を下回る最小 N**（それ以上のすべての N でも下回り続けるもの）をユニット別に出す。
あわせて「ラベルなし N 例はラベルあり何例分に相当するか」を、ラベルあり曲線の平均 ECE を log N で補間して読む。

In [9]:
safe = {m: om.min_safe_n(summ, m, harm_threshold) for m in ("om", "ratio", "label")}
safe_tol = {m: om.min_safe_n(summ, m, harm_threshold, "p_harm_tol") for m in ("om", "ratio", "label")}
safe_df = pd.concat({"P(害)": pd.DataFrame(safe), "P(害>0.01)": pd.DataFrame(safe_tol)}, axis=1).reindex(UNITS)
display(Markdown(f"**P(害) < {harm_threshold:.0%} となる最小 N**（None = 試した N では満たさない）")); display(safe_df)
equiv = {u: om.label_equivalent(summ, u, "om") for u in UNITS}
equiv_df = pd.DataFrame(equiv).T.reindex(UNITS)
display(Markdown("**ラベルなし N 例（出力整合）＝ラベルあり何例分か**（列 = ラベルなしの N）")); display(equiv_df)

**P(害) < 10% となる最小 N**（None = 試した N では満たさない）

P(害)             P(害>0.01)              
                   om ratio label        om  ratio  label
LOSO-A          100.0   NaN   NaN      30.0  200.0  100.0
LOSO-B           20.0  10.0  20.0      20.0   10.0   20.0
field 3T          NaN   NaN   NaN       NaN    NaN    NaN
vendor Philips    NaN   NaN   NaN       NaN    NaN    NaN
in-dist           NaN   NaN   NaN       NaN    NaN    NaN

**ラベルなし N 例（出力整合）＝ラベルあり何例分か**（列 = ラベルなしの N）

,5,10,20,30,50,100,200
LOSO-A,7,20,30,69,>=100,>=100,>=100
LOSO-B,8,16,44,61,88,>=200,>=200
field 3T,8,24,>=50,>=50,>=50,24,NaN
vendor Philips,10,21,29,>=30,>=30,5,NaN
in-dist,8,15,40,62,>=100,>=100,23


## 3. 評価 3：頑健性

全例（評価 1 の設定）で、target 側の疑い回答だけを壊す。source 側（train の病理 grade、val の π）はそのまま。

- **誤分類**：周辺保存フリップ 15%（grade_prior_eval の水準 Cb と同じ）を 50 回
- **欠損**：疑い回答の 0 / 25 / 50 / 100% を「不明」に置換 → その症例は年齢のみの π（25 / 50% は 50 回）

In [10]:
rob = om.robustness(runs, flip=flip_rate, missing=missing, reps=reps, seed=seed)
rob["unit"] = pd.Categorical(rob["unit"], UNITS, ordered=True)
clean = rob[(rob.cond == "missing") & (rob.level == 0.0)].groupby(["unit", "method"], observed=True)["ece"].mean()
def cond_table(cond):
    r = rob[rob.cond == cond]
    per_run = r.groupby(["unit", "method", "level", "run"], observed=True)["ece"].mean().reset_index()
    t = per_run.groupby(["unit", "method", "level"], observed=True)["ece"].mean().unstack("level")
    return t
flip_t = rob[rob.cond == "flip"].groupby(["unit", "method"], observed=True)["ece"].agg(
    mean="mean", lo=lambda x: x.quantile(.025), hi=lambda x: x.quantile(.975))
flip_t["clean"] = clean; flip_t["劣化"] = flip_t["mean"] - flip_t["clean"]
display(Markdown(f"**誤分類 {flip_rate:.0%}（周辺保存）**")); display(flip_t.round(3))
miss_t = cond_table("missing")
display(Markdown("**疑い欠損率ごとの ECE**（列 = 不明にした割合）")); display(miss_t.round(3))

**誤分類 15%（周辺保存）**

mean     lo     hi  clean     劣化
unit           method                                     
LOSO-A         b_ratio   0.081  0.046  0.114  0.070  0.012
               c_om_cov  0.039  0.024  0.057  0.038  0.001
LOSO-B         b_ratio   0.068  0.037  0.091  0.064  0.005
               c_om_cov  0.048  0.036  0.063  0.042  0.006
field 3T       b_ratio   0.099  0.064  0.125  0.090  0.009
               c_om_cov  0.081  0.060  0.104  0.070  0.011
vendor Philips b_ratio   0.087  0.053  0.132  0.092 -0.005
               c_om_cov  0.093  0.065  0.130  0.092  0.001
in-dist        b_ratio   0.045  0.028  0.063  0.044  0.001
               c_om_cov  0.046  0.029  0.063  0.044  0.001

**疑い欠損率ごとの ECE**（列 = 不明にした割合）

level                     0.00   0.25   0.50   1.00
unit           method                              
LOSO-A         b_ratio   0.070  0.070  0.070  0.069
               c_om_cov  0.038  0.037  0.037  0.037
LOSO-B         b_ratio   0.064  0.069  0.080  0.105
               c_om_cov  0.042  0.049  0.056  0.090
field 3T       b_ratio   0.090  0.091  0.092  0.091
               c_om_cov  0.070  0.074  0.074  0.074
vendor Philips b_ratio   0.092  0.089  0.087  0.089
               c_om_cov  0.092  0.091  0.093  0.100
in-dist        b_ratio   0.044  0.044  0.044  0.043
               c_om_cov  0.044  0.044  0.044  0.045

### LOSO-B の b̂ の内訳

b̂（出力整合）− b（比方式）= 共変量では説明できず、モデル出力の測定で初めて拾われた分。
実験 A の綱引き（撮像が criterion を押し下げる）が正しければ、**正**になる（押し下げを打ち返す向き）。

In [11]:
dec = full[full.method.isin(["b_ratio", "c_om_cov", "e_om_oracle"])].pivot_table(
    index=["unit", "run"], columns="method", values="b", observed=True)
dec["om - ratio"] = dec["c_om_cov"] - dec["b_ratio"]
dec["oracle - om"] = dec["e_om_oracle"] - dec["c_om_cov"]
display(dec.round(3))
decu = dec.groupby("unit", observed=True).agg(["mean", "std"]).round(3)
display(decu)

method                    b_ratio  c_om_cov  e_om_oracle  om - ratio  oracle - om
unit           run                                                               
LOSO-A         foldA_s0     0.125     1.151        1.217       1.025        0.067
               foldA_s1     0.125     0.466        0.537       0.341        0.071
               foldA_s2     0.125     0.708        0.777       0.582        0.069
LOSO-B         foldB_s0     1.065     1.134        1.174       0.069        0.040
               foldB_s1     1.065     1.507        1.547       0.442        0.039
               foldB_s2     1.065     1.604        1.645       0.538        0.042
field 3T       field_s0    -0.334     0.722        0.816       1.056        0.093
               field_s1    -0.334    -0.197       -0.120       0.137        0.077
               field_s2    -0.334     0.068        0.159       0.402        0.091
vendor Philips vendor_s0    0.283    -0.114        0.182      -0.397        0.296
               vendor_s1    0.283     0.019        0.301      -0.264        0.282
               vendor_s2    0.283     0.242        0.580      -0.041        0.338
in-dist        rand0_s0    -0.146    -0.589       -0.368      -0.443        0.221
               rand0_s1    -0.146     0.090        0.287       0.235        0.198
               rand0_s2    -0.146    -0.843       -0.593      -0.698        0.250
               rand1_s0     0.100     0.105        0.102       0.005       -0.004
               rand1_s1     0.100     0.316        0.312       0.215       -0.003
               rand1_s2     0.100    -0.004       -0.007      -0.104       -0.003
               rand2_s0     0.074     0.413        0.257       0.338       -0.155
               rand2_s1     0.074    -0.224       -0.365      -0.299       -0.140
               rand2_s2     0.074     0.338        0.210       0.264       -0.129
               rand3_s0     0.124     0.326        0.217       0.203       -0.110
               rand3_s1     0.124     0.144        0.023       0.020       -0.121
               rand3_s2     0.124     0.112        0.025      -0.012       -0.087
               rand4_s0     0.014     0.391        0.455       0.377        0.064
               rand4_s1     0.014    -0.278       -0.220      -0.292        0.058
               rand4_s2     0.014     0.214        0.275       0.200        0.062

method         b_ratio      c_om_cov        e_om_oracle        om - ratio        oracle - om       
                  mean  std     mean    std        mean    std       mean    std        mean    std
unit                                                                                               
LOSO-A           0.125  0.0    0.775  0.347       0.844  0.345      0.649  0.347       0.069  0.002
LOSO-B           1.065  0.0    1.415  0.248       1.455  0.249      0.350  0.248       0.040  0.001
field 3T        -0.334  0.0    0.198  0.473       0.285  0.480      0.532  0.473       0.087  0.009
vendor Philips   0.283  0.0    0.049  0.180       0.354  0.205     -0.234  0.180       0.305  0.029
in-dist          0.033  0.1    0.034  0.369       0.041  0.302      0.001  0.314       0.007  0.135

## 4. 処方箋の階段

ユニット別の ECE を、必要な情報量の順に並べる。出力整合（N 例）は評価 2 で P(害) < 10% を満たした最小 N の値、
ラベル 20 例は評価 2 の N=20（どちらも残り症例での ECE。全例版とは母集団が少し違う）。
EM は既報で失敗（LOSO-A 0.082 → 0.141）なので、ここでは再計算しない。

In [12]:
def ev1(m): return {u: ece(u, m) for u in UNITS}
def ev2(m, n_of):
    out = {}
    for u in UNITS:
        n = n_of(u)
        r = summ[(summ.unit == u) & (summ.method == m) & (summ.n == n)] if n else summ.iloc[:0]
        out[u] = float(r["ece_mean"].iloc[0]) if len(r) else np.nan
    return out
ladder = pd.DataFrame({
    "補正なし": ev1("a_raw"),
    "EM": {u: np.nan for u in UNITS},
    "年齢のみ（比方式）": ev1("b0_ratio_age"),
    "年齢+疑い（比方式）": ev1("b_ratio"),
    "年齢+疑い（出力整合・全例）": ev1("c_om_cov"),
    "年齢+疑い（出力整合・安全 N 例）": ev2("om", lambda u: safe["om"][u]),
    "ラベル 20 例": ev2("label", lambda u: 20),
    "oracle（全例ラベル）": ev1("f_label_mle"),
}).T
ladder["安全 N（出力整合）"] = None
ladder.loc["年齢+疑い（出力整合・安全 N 例）", "安全 N（出力整合）"] = str(safe["om"])
display(ladder.round(3))

,LOSO-A,LOSO-B,field 3T,vendor Philips,in-dist,安全 N（出力整合）
補正なし,0.082,0.162,0.088,0.085,0.044,None
EM,NaN,NaN,NaN,NaN,NaN,None
年齢のみ（比方式）,0.074,0.105,0.091,0.096,0.045,None
年齢+疑い（比方式）,0.070,0.064,0.090,0.092,0.044,None
年齢+疑い（出力整合・全例）,0.038,0.042,0.070,0.092,0.044,None
年齢+疑い（出力整合・安全 N 例）,0.049,0.065,NaN,NaN,NaN,"{'LOSO-A': 100, 'LOSO-B': 20, 'field 3T': None..."
ラベル 20 例,0.075,0.078,0.108,0.115,0.067,None
oracle（全例ラベル）,0.042,0.045,0.078,0.086,0.043,None


## 5. 保存

- `results/output_matching.json`：評価 1 の全行、評価 2 の要約、評価 3 の要約、検算
- `results/output_matching.md`：Discord 用の要約
- `results/figures/output_matching_n.{png,svg}`：評価 2 の図

In [13]:
def recs(df): return json.loads(df.reset_index().to_json(orient="records", force_ascii=False))
res = {
    "params": {"reps": reps, "seed": seed, "ns": ns, "flip_rate": flip_rate, "missing": missing,
               "harm_threshold": harm_threshold, "bracket": list(om.BRACKET)},
    "checks": checks,
    "eval1_per_run": recs(full.astype({"unit": str, "method": str})),
    "eval1_unit": recs(t1.astype({"unit": str, "method": str})),
    "prereg": recs(prereg),
    "eval2_summary": recs(summ.astype({"unit": str})),
    "eval2_min_safe_n": safe,
    "eval2_min_safe_n_tol": safe_tol,
    "eval2_label_equivalent": equiv,
    "eval3_flip": recs(flip_t.reset_index().astype({"unit": str})),
    "eval3_missing": recs(miss_t.reset_index().astype({"unit": str}).rename(columns=str)),
    "eval3_decomposition": recs(dec.reset_index().astype({"unit": str})),
    "ladder": recs(ladder.rename(columns=str)),
}
Path(out_json).write_text(json.dumps(res, indent=2, ensure_ascii=False, default=float))

def f3(x): return "—" if pd.isna(x) else f"{x:.3f}"
L = []
L.append("# 出力整合（output matching）— 結果要約\n")
L.append(f"検算：e（oracle 係留）と f（全例ラベル MLE）の差 最大 {checks['max_abs_e_minus_f']:.1e}、"
         f"ΔAUC 最大 {checks['max_abs_dAUC']:.1e}、clip {len(checks['clipped_eval1'])} 件。"
         f"LOSO-B の π̂_cov = {checks['pi_cov_test_lossb']*100:.1f}%\n")
L.append("## 評価 1：全例 ECE（ラン平均）\n```")
hdr = f"{'手法':<20}" + "".join(f"{u[:10]:>11}" for u in UNITS)
L.append(hdr)
for m in MORDER:
    L.append(f"{om.METHODS_FULL[m]:<20}" + "".join(f"{ece(u, m):>11.3f}" for u in UNITS))
L.append(f"{'b̂（出力整合 c）':<20}" + "".join(f"{bb(u, 'c_om_cov'):>+11.2f}" for u in UNITS))
L.append(f"{'b̂（全例ラベル f）':<20}" + "".join(f"{bb(u, 'f_label_mle'):>+11.2f}" for u in UNITS))
L.append("```\n")
L.append("事前予測との照合：")
for _, r in prereg.iterrows():
    L.append(f"- {r['unit']}：予測 {r['予測']} → 実測 {r['出力整合 c']:.3f}（b̂ {r['b̂_c']:+.2f}）**{r['判定']}**")
L.append("\n## 評価 2：N 依存性（ECE 平均 / P(害) / P(害>0.01)）\n```")
for u in UNITS:
    L.append(f"[{u}]")
    L.append(f"{'N':>5}{'raw':>7}{'出力整合':>20}{'比方式':>20}{'ラベルあり':>20}")
    s = summ[summ.unit == u]
    for n in ns:
        cells = []
        for m in ("om", "ratio", "label"):
            r = s[(s.method == m) & (s.n == n)]
            cells.append("—" if r.empty else
                         f"{r.ece_mean.iloc[0]:.3f}/{r.p_harm.iloc[0]*100:>3.0f}%/{r.p_harm_tol.iloc[0]*100:>3.0f}%")
        if all(c == "—" for c in cells): continue
        raw = s[(s.method == "raw") & (s.n == n)].ece_mean
        L.append(f"{n:>5}{raw.iloc[0]:>7.3f}" + "".join(f"{c:>20}" for c in cells))
L.append("```\n")
L.append("（field / vendor の N=100 は残り 95 / 40 例で ECE が小標本バイアスで跳ねる。手法比較には使えない）\n")
L.append(f"P(害) < {harm_threshold:.0%} の最小 N（出力整合）：" +
         "、".join(f"{u} {safe['om'][u]}" for u in UNITS))
L.append(f"P(害>0.01) < {harm_threshold:.0%} の最小 N（出力整合）：" +
         "、".join(f"{u} {safe_tol['om'][u]}" for u in UNITS))
L.append("ラベルなし N 例 ＝ ラベルあり何例分：" +
         "；".join(f"{u} " + ", ".join(f"{k}→{v}" for k, v in equiv[u].items()) for u in UNITS[:4]) + "\n")
L.append("## 評価 3：頑健性（全例、ECE）\n```")
L.append(f"{'':<16}{'clean':>8}{'flip15%':>9}{'miss25':>8}{'miss50':>8}{'miss100':>8}")
for u in UNITS:
    for m, nm in (("b_ratio", "比方式"), ("c_om_cov", "出力整合")):
        ft = flip_t.loc[(u, m)]; mt = miss_t.loc[(u, m)]
        L.append(f"{u[:8]+' '+nm:<16}{ft['clean']:>8.3f}{ft['mean']:>9.3f}{mt[0.25]:>8.3f}{mt[0.5]:>8.3f}{mt[1.0]:>8.3f}")
L.append("```\n")
d = dec.loc["LOSO-B"]
L.append(f"LOSO-B の内訳：b̂ 出力整合 {d['c_om_cov'].mean():+.2f} = 比方式 {d['b_ratio'].mean():+.2f}"
         f" ＋ 共変量で説明できない分 {d['om - ratio'].mean():+.2f}（seed 間 sd {d['om - ratio'].std():.2f}）\n")
L.append("## 処方箋の階段（ECE）\n```")
L.append(f"{'':<28}" + "".join(f"{u[:10]:>11}" for u in UNITS))
for k, r in ladder.iterrows():
    L.append(f"{k:<28}" + "".join(f"{f3(r[u]):>11}" for u in UNITS))
L.append("```")
Path(out_md).write_text("\n".join(L) + "\n")
print("\n".join(L))

# 出力整合（output matching）— 結果要約

検算：e（oracle 係留）と f（全例ラベル MLE）の差 最大 3.1e-08、ΔAUC 最大 0.0e+00、clip 0 件。LOSO-B の π̂_cov = 27.9%

## 評価 1：全例 ECE（ラン平均）
```
手法                       LOSO-A     LOSO-B   field 3T vendor Phi    in-dist
補正なし                      0.082      0.162      0.088      0.085      0.044
比方式（年齢のみ）                 0.074      0.105      0.091      0.096      0.045
比方式（年齢+疑い）                0.070      0.064      0.090      0.092      0.044
出力整合（年齢+疑い）               0.038      0.042      0.070      0.092      0.044
出力整合（年齢のみ）                0.037      0.090      0.074      0.100      0.045
出力整合（oracle 係留）           0.042      0.045      0.078      0.086      0.043
全例ラベル切片                   0.042      0.045      0.078      0.086      0.043
b̂（出力整合 c）                +0.77      +1.42      +0.20      +0.05      +0.03
b̂（全例ラベル f）               +0.84      +1.46      +0.28      +0.35      +0.04
```

事前予測との照合：
- LOSO-B：予測 0.046–0.052 → 実測 0.042（b̂ +1.42）**予測より良い**
- LOSO-A：予測 0.045–0.0